In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import unicodedata
from pathlib import Path

import pandas as pd
import transformers
from transformers import AutoTokenizer

DATA_DIR = Path("/content/drive/MyDrive/greek-news-nlp")
MODEL_DIR = DATA_DIR / "greek-bert-news-classifier"

test = pd.read_json(DATA_DIR / "el_test.jsonl", lines=True)
tok = AutoTokenizer.from_pretrained(MODEL_DIR)

def prep(text):
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        c for c in text if unicodedata.category(c) != "Mn"
    )
    return text.lower()

texts = test["text_clean"].tolist()
assert all(isinstance(t, str) for t in texts)

raw_ids = tok(
    texts, truncation=True, max_length=256
)["input_ids"]

prep_ids = tok(
    [prep(t) for t in texts],
    truncation=True,
    max_length=256,
)["input_ids"]

different = [
    i for i, (a, b) in enumerate(zip(raw_ids, prep_ids))
    if a != b
]

print("Transformers:", transformers.__version__)
print("Tokenizer:", type(tok).__name__)
print("Test articles:", len(test))
print("Unique IDs:", test["document_id"].nunique())
print("Articles with different tokens:", len(different))

if different:
    i = different[0]
    print("\nExample document_id:", test.iloc[i]["document_id"])
    print("Without prep:", tok.convert_ids_to_tokens(raw_ids[i])[:40])
    print("With prep:   ", tok.convert_ids_to_tokens(prep_ids[i])[:40])

Transformers: 5.18.0
Tokenizer: BertTokenizer
Test articles: 788
Unique IDs: 788
Articles with different tokens: 81

Example document_id: macocu.el.9598177
Without prep: ['[CLS]', '"', 'γ', '.', 'σ', '.', 'kal', '##ama', '##ta', 'running', 'project', '"', ':', 'δυναμικη', 'παρουσια', 'στον', 'ημιμαραθωνιο', 'καλαματας', 'δυναμικη', 'παρουσια', 'με', 'νεους', 'αθλητες', 'και', 'ρεκορ', 'ειχε', 'ο', 'γ', '.', 'σ', '.', 'kal', '##ama', '##ta', 'running', 'project', 'στον', '3', '##ο', 'ημιμαραθωνιο']
With prep:    ['[CLS]', '"', 'γ', '.', 'ς', '.', 'kal', '##ama', '##ta', 'running', 'project', '"', ':', 'δυναμικη', 'παρουσια', 'στον', 'ημιμαραθωνιο', 'καλαματας', 'δυναμικη', 'παρουσια', 'με', 'νεους', 'αθλητες', 'και', 'ρεκορ', 'ειχε', 'ο', 'γ', '.', 'ς', '.', 'kal', '##ama', '##ta', 'running', 'project', 'στον', '3', '##ο', 'ημιμαραθωνιο']


In [4]:
import torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
).to(device).eval()

y_true = test["GPT-IPTC-label"].astype(str).tolist()
labels = sorted(model.config.id2label.values())
assert set(y_true).issubset(set(labels))

@torch.inference_mode()
def predict(texts, batch_size=8):
    predictions = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tok(
            batch,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt",
        ).to(device)

        indices = model(**inputs).logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(model.config.id2label[i] for i in indices)

    return predictions

# Ίδιο μοντέλο, batch size και precision και στις δύο περιπτώσεις.
pred_raw = predict(texts)
pred_prep = predict([prep(t) for t in texts])

for name, predictions in [
    ("Without prep", pred_raw),
    ("With prep", pred_prep),
]:
    print(
        name,
        "| Accuracy:", round(accuracy_score(y_true, predictions), 4),
        "| Macro-F1:", round(
            f1_score(
                y_true, predictions,
                labels=labels, average="macro", zero_division=0,
            ),
            4,
        ),
    )

changed = sum(a != b for a, b in zip(pred_raw, pred_prep))
print("Articles with different predictions:", changed)

audit = pd.DataFrame({
    "document_id": test["document_id"],
    "true": y_true,
    "pred_without_prep": pred_raw,
    "pred_with_prep": pred_prep,
})

Device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Without prep | Accuracy: 0.8338 | Macro-F1: 0.775
With prep | Accuracy: 0.8363 | Macro-F1: 0.7776
Articles with different predictions: 2


In [5]:
import hashlib
import json
import platform
from datetime import datetime, timezone

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

output_dir = DATA_DIR / "greekbert_audit"
output_dir.mkdir(exist_ok=True)

audit.to_csv(output_dir / "predictions.csv", index=False)

model_files = sorted(
    p for p in MODEL_DIR.iterdir()
    if p.is_file() and (
        p.suffix in {".json", ".safetensors"}
        or p.name in {"pytorch_model.bin", "vocab.txt"}
    )
)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "device": str(device),
    "test_sha256": sha256_file(DATA_DIR / "el_test.jsonl"),
    "model_files_sha256": {
        p.name: sha256_file(p) for p in model_files
    },
}

with open(output_dir / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved to:", output_dir)
print("Model files:", list(manifest["model_files_sha256"]))

Saved to: /content/drive/MyDrive/greek-news-nlp/greekbert_audit
Model files: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
